In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

# Walk-forward 再学習（Phase E）

OCO戦略改善 Work Plan § Phase E。test期間（2024-03〜2026-04）を9チャンク（≈四半期）に
分割し、各チャンクの直前1年分で lstm_focal をウォームスタートからfine-tuneして
そのチャンクの p_move を予測する。

出力（/kaggle/working/）:
- p_move_wf_test.npy: walk-forward 予測（test全長）
- p_move_wf_valtail.npy: 各チャンク直前 VAL_TAIL 本の予測（閾値選択用、shape (9, VAL_TAIL)）
- wf_chunks.npy: チャンク境界（test内インデックス）

In [ ]:
import subprocess, sys

# Kaggle プリインストールの torch (2.10+cu128) は P100 (sm_60) 非対応のため
# sm_60 / sm_75 両対応の torch==2.5.1+cu121 を明示固定（バージョン無指定はno-op）
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
import torch
assert torch.__version__.startswith("2.5.1"), f"torch downgrade failed: {torch.__version__}"
print(f"torch {torch.__version__}")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

DATASET_DIR = Path("/kaggle/input/datasets/nomuhosokawa/pearless-usdjpy-m5")
WORKING_DIR = Path("/kaggle/working")
if not DATASET_DIR.exists():
    DATASET_DIR = Path("/kaggle/input/pearless-usdjpy-m5")

for _repo_candidate in [
    Path("/kaggle/input/pearless-src"),
    Path("/kaggle/input/datasets/nomuhosokawa/pearless-src"),
]:
    if _repo_candidate.exists():
        REPO_ROOT = _repo_candidate
        break
else:
    REPO_ROOT = Path("..")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

for _aux in [
    Path("/kaggle/input/pearless-aux-labels"),
    Path("/kaggle/input/datasets/nomuhosokawa/pearless-aux-labels"),
]:
    if _aux.exists():
        AUX_DIR = _aux
        break
else:
    AUX_DIR = Path("data/labels")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, Dataset: {DATASET_DIR}, Aux: {AUX_DIR}")

In [ ]:
# 全期間を時系列順に連結（train → val → test）
X_all = np.concatenate([
    np.load(DATASET_DIR / "X_train.npy").astype(np.float32),
    np.load(DATASET_DIR / "X_val.npy").astype(np.float32),
    np.load(DATASET_DIR / "X_test.npy").astype(np.float32),
])
y_all = np.concatenate([
    np.load(DATASET_DIR / "y_train.npy"),
    np.load(DATASET_DIR / "y_val.npy"),
    np.load(DATASET_DIR / "y_test.npy"),
])
n_test = len(np.load(DATASET_DIR / "y_test.npy"))
TEST_START = len(y_all) - n_test
print(f"X_all: {X_all.shape}, TEST_START={TEST_START}, n_test={n_test}")

In [ ]:
# Walk-forward ループ
from models.configs import MODEL_CONFIGS
from models.training import train

N_CHUNKS = 9
FT_WIN = 105120      # fine-tune に使う直前ウィンドウ数（約1年）
VAL_TAIL = 15000     # early stopping / 閾値選択用の直前スライス
FT_EPOCHS = 8
FT_PATIENCE = 3

cfg = MODEL_CONFIGS["lstm_focal"]
model = cfg.build_model()
model.load_state_dict(torch.load(AUX_DIR / "warmstart_lstm_focal.pt",
                                 map_location="cpu", weights_only=True))

edges = np.linspace(0, n_test, N_CHUNKS + 1, dtype=np.int64)
p_move_test = np.zeros(n_test, dtype=np.float32)
p_move_valtail = np.zeros((N_CHUNKS, VAL_TAIL), dtype=np.float32)


def predict_pmove(m, X):
    m.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(DEVICE)
            p = m(xb).cpu().numpy()
            out.append(p[:, 0] + p[:, 1])
    return np.concatenate(out)


for q in range(N_CHUNKS):
    s = TEST_START + int(edges[q])
    e = TEST_START + int(edges[q + 1])
    ft_lo = s - FT_WIN
    X_ft, y_ft = X_all[ft_lo : s - VAL_TAIL], y_all[ft_lo : s - VAL_TAIL]
    X_vt, y_vt = X_all[s - VAL_TAIL : s], y_all[s - VAL_TAIL : s]

    ckpt_dir = WORKING_DIR / f"wf_q{q}"
    train(
        model=model,
        X_train=X_ft, y_train=y_ft,
        X_val=X_vt, y_val=y_vt,
        model_name=f"wf_q{q}",
        n_epochs=FT_EPOCHS,
        patience=FT_PATIENCE,
        lr=cfg.train.lr,
        batch_size=cfg.train.batch_size,
        weight_decay=cfg.train.weight_decay,
        scheduler_t0=cfg.train.scheduler_t0,
        loss_type=cfg.train.loss_type,
        focal_gamma=cfg.train.focal_gamma,
        early_stop_metric="val_f1_updown",
        checkpoint_dir=str(ckpt_dir),
        log_dir=str(WORKING_DIR / "logs"),
    )
    # fine-tune 中のベスト重みで予測し、次チャンクへもその重みから継続
    model.load_state_dict(torch.load(ckpt_dir / "best_model.pt",
                                     map_location="cpu", weights_only=True))
    p_move_valtail[q] = predict_pmove(model, X_vt)
    p_move_test[int(edges[q]):int(edges[q+1])] = predict_pmove(model, X_all[s:e])
    print(f"chunk {q}: windows [{s}, {e}) 完了")

np.save(WORKING_DIR / "p_move_wf_test.npy", p_move_test)
np.save(WORKING_DIR / "p_move_wf_valtail.npy", p_move_valtail)
np.save(WORKING_DIR / "wf_chunks.npy", edges)
print("walk-forward 完了")

In [ ]:
# 出力確認
for f in ["p_move_wf_test.npy", "p_move_wf_valtail.npy", "wf_chunks.npy"]:
    a = np.load(WORKING_DIR / f)
    print(f, a.shape, getattr(a, 'dtype', None))
assert (np.load(WORKING_DIR / "p_move_wf_test.npy") > 0).all(), "未予測領域が残っている"
print("OK")